In [1]:
import pandas as pd
from engine import load_env, create_engine_from_env
from models.manufacture_module import get_manufacturer_df, get_active_unique_manufacturers, get_manufacturer_bulletins_json, print_bulletin_details, convert_bulletin_to_df, save_vehicle_models_to_csv, batch_save_manufacturer_models, search_models_by_description



In [2]:
import json
from typing import Union, Any

def parse_json_string(json_string: str) -> Union[dict, list]:
    """
    Parse a JSON string into a Python object (dict or list).

    Args:
        json_string (str): A valid JSON string.

    Returns:
        dict or list: Parsed JSON object.

    Raises:
        ValueError: If the input is not valid JSON.
        TypeError: If input is not a string.
    """
    if not isinstance(json_string, str):
        raise TypeError("Input must be a JSON string")

    try:
        return json.loads(json_string)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON string: {exc}") from exc


In [3]:

# Loaded environment variables and created database engine
load_env()
engine = create_engine_from_env()


In [4]:

# Batch save vehicle models for specified manufacturers
# manufactures = ["Hyundai","Honda","Kia","Mazda","Subaru","Toyota","Volkswagen"]
# manufactures = ["Genesis"]
# batch_save_manufacturer_models(engine, manufactures)


In [5]:

model_number_db = pd.read_csv("db/db_vehicle_models.csv")


In [6]:
make_df_genesis = model_number_db[model_number_db["Manufacturer"] == "Genesis"]
make_df_hyundai = model_number_db[model_number_db["Manufacturer"] == "Hyundai"]

In [7]:



# make_df_hyundai[
    
#     (make_df_hyundai["ModelYear"]==2024)
#     # (1==1)
#     &(make_df_hyundai["Description"].str.contains("Es", case=False, na=False))
#     # # & (make_df_hyundai["Description"].str.contains("Cross", case=False, na=False))
#     # # & (make_df_hyundai["Description"].str.contains("noir", case=False, na=False))
#     # & (make_df_hyundai["Package"]=="KE00")
#     ]

In [9]:
# make_df.columns

In [26]:

make_df_hyundai[
        (1==1) 
        & (make_df_hyundai["ModelYear"]==2026)
        & make_df_hyundai["Description"].str.contains("kona", case=False, na=False)
        & make_df_hyundai["Description"].str.contains("electric", case=False, na=False)
        # & make_df_hyundai["Description"].str.contains("electric", case=False, na=False)

        # & make_df_hyundai["Description"].str.contains("phev", case=False, na=False)
        # & make_df_hyundai["Description"].str.contains("noir", case=False, na=False)
    ]
 

,Manufacturer,ModelYear,ModelNumber,Description,Description2,Package,Style_ID
942,Hyundai,2026,KEEW5ZE4PR00,Kona Electric Preferred,NaN,NaN,NaN
943,Hyundai,2026,KEEW5ZE4PRTR,Kona Electric Preferred w/ Trend Pkg,NaN,NaN,NaN
944,Hyundai,2026,KEEW5ZE4PRUP,Kona Electric Pref w/Ult Pkg,NaN,NaN,NaN
945,Hyundai,2026,KEEW5ZE4PRUT,Kona Electric Pref w/Ult Pkg Two Tone,NaN,NaN,NaN


In [28]:

make = "Hyundai"
year = 2024
keywords = ['elantra', 'ess']
# keywords = ['elantra', 'hev']
# keywords = ['elantra', 'lux']
# keywords = ['elantra', 'n']
# keywords = ['elantra', 'n-line']
# keywords = ['elantra', 'pref']
# keywords = ['elantra', 'ult']
# keywords =  ['ioniq', '5', 'lux']
# keywords =   ['ioniq', '5', 'pref']
# keywords =   ['ioniq', '5', 'ult']
# keywords =   ['ioniq', '6', 'pref']
# keywords =   ['ioniq', '6', 'ult']
# keywords =   ['kona', 'ess']
# keywords =   ['Kona', 'n-line']
# keywords =   ['Kona', 'n-line', 'ult']
# keywords =   ['kona', 'trend']
# keywords =    ['kona', 'ev', 'ult']


search_models_by_description(make, year, keywords)
 

,Manufacturer,ModelYear,ModelNumber,Description,Description2,Package,Style_ID
380,Hyundai,2024,ELCS4V2BES00,Elantra Essential IVT,NaN,NaN,NaN
515,Hyundai,2024,EL74IF20A100,Elantra Essential IVT,NaN,NaN,NaN


I want us to make a change to the model look up.

We are getting miss-match in some instances where data in the csv db is not standardized to meet the expected keyword standard in the translator targets. This means that while the translator is translating, some keywords in teh csv db are still in the previous form. for instance, the translator is set to take ult -> ultimate. however some db records have utl which make the search to fail.  

The solution that I was thinking, add a step on the db data pull utility.

The process that i am thinking is as follows:

1. Pull the db,

2. Clean up the db. 

3. Standardize keywords: based on translator_keywors









I want to add an edge case that we need to handle:

There are some vehicles that have 2 model numbers, the old model and the new model. The characteristic, they are ht esame mafucture, year and the description is the same. The current logic states that if the search_model utility gets two records, it should flag as ambigous, but for this one, I wna us to add another condition, that if they are more than one option, but the year and description are the same, do the following:

Duplicate the rows for that search key
and create a records for each of the model numbers.

Fore example:

If we pass in :
make = "Hyundai"
year = 2024
keywords = ['elantra', 'ess']

We'll get the following two records:

380	Hyundai	2024	ELCS4V2BES00	Elantra Essential IVT			
515	Hyundai	2024	EL74IF20A100	Elantra Essential IVT			


At the end we'll have a list of parts records with ELCS4V2BES00 and another one with EL74IF20A100 as the model number.

I want this feature to be given a flag that will enable or disable it inside the OEM config. This way we can control it better. 

Give me a plan on how to do empliment this.


My 